# Caching

In [1]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "10g")
    .config("spark.sql.files.maxPartitionBytes", "268435456")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .master("local[*]")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

# spark = SparkSession.builder.appName('MyApp').getOrCreate()
# sc = spark.sparkContext

In [3]:
spark

In [5]:
customers_file = "../data/data_skew/customers.parquet"
df_customers = spark.read.parquet(customers_file)

In [6]:
df_customers.show(5, False)

+----------+-------------+---+------+----------+-----+-----------+
|cust_id   |name         |age|gender|birthday  |zip  |city       |
+----------+-------------+---+------+----------+-----+-----------+
|C007YEYTX9|Aaron Abbott |34 |Female|7/13/1991 |97823|boston     |
|C00B971T1J|Aaron Austin |37 |Female|12/16/2004|30332|chicago    |
|C00WRSJF1Q|Aaron Barnes |29 |Female|3/11/1977 |23451|denver     |
|C01AZWQMF3|Aaron Barrett|31 |Male  |7/9/1998  |46613|los_angeles|
|C01BKUFRHA|Aaron Becker |54 |Male  |11/24/1979|40284|san_diego  |
+----------+-------------+---+------+----------+-----+-----------+
only showing top 5 rows



In [9]:
df_base = (
    df_customers
    .filter(col("city") == "boston")
    .withColumn(
        "customer_group", 
        when(
            col("age").between(20, 30), 
            lit("young") 
        )
        .when(
            col("age").between(31, 50), 
            lit("mid") 
        )
        .when(
            col("age") > 51, 
            lit("old") 
        )
        .otherwise(lit("kid"))
     )
    .select("cust_id", "name", "age", "gender", "birthday", "zip", "city", "customer_group")
)

df_base.cache() 
df_base.show(5, False)

+----------+--------------+---+------+---------+-----+------+--------------+
|cust_id   |name          |age|gender|birthday |zip  |city  |customer_group|
+----------+--------------+---+------+---------+-----+------+--------------+
|C007YEYTX9|Aaron Abbott  |34 |Female|7/13/1991|97823|boston|mid           |
|C08XAQUY73|Aaron Lambert |54 |Female|11/5/1966|75218|boston|old           |
|C094P1VXF9|Aaron Lindsey |24 |Male  |9/21/1990|29399|boston|young         |
|C097SHE1EF|Aaron Lopez   |22 |Female|4/18/2001|82129|boston|young         |
|C0DTC6436T|Aaron Schwartz|52 |Female|7/9/1962 |57192|boston|old           |
+----------+--------------+---+------+---------+-----+------+--------------+
only showing top 5 rows



In [10]:
df1 = (
    df_base
    .withColumn("test_column_1", lit("test_column_1"))
    .withColumn("birth_year", split("birthday", "/").getItem(2))
)

df1.explain(True)
df1.show(5, False)

== Parsed Logical Plan ==
'Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, customer_group#57, test_column_1#307, split('birthday, /, -1)[2] AS birth_year#317]
+- Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, customer_group#57, test_column_1 AS test_column_1#307]
   +- Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, customer_group#57]
      +- Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, CASE WHEN ((cast(age#45 as int) >= 20) AND (cast(age#45 as int) <= 30)) THEN young WHEN ((cast(age#45 as int) >= 31) AND (cast(age#45 as int) <= 50)) THEN mid WHEN (cast(age#45 as int) > 51) THEN old ELSE kid END AS customer_group#57]
         +- Filter (city#49 = boston)
            +- Relation [cust_id#43,name#44,age#45,gender#46,birthday#47,zip#48,city#49] parquet

== Analyzed Logical Plan ==
cust_id: string, name: string, age: string, gender: string, birthday: string, zip:

In [11]:
df2 = (
    df_base
    .withColumn("test_column_2", lit("test_column_2"))
    .withColumn("birth_month", split("birthday", "/").getItem(1))
)

df2.explain(True)
df2.show(5, False)

== Parsed Logical Plan ==
'Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, customer_group#57, test_column_2#649, split('birthday, /, -1)[1] AS birth_month#659]
+- Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, customer_group#57, test_column_2 AS test_column_2#649]
   +- Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, customer_group#57]
      +- Project [cust_id#43, name#44, age#45, gender#46, birthday#47, zip#48, city#49, CASE WHEN ((cast(age#45 as int) >= 20) AND (cast(age#45 as int) <= 30)) THEN young WHEN ((cast(age#45 as int) >= 31) AND (cast(age#45 as int) <= 50)) THEN mid WHEN (cast(age#45 as int) > 51) THEN old ELSE kid END AS customer_group#57]
         +- Filter (city#49 = boston)
            +- Relation [cust_id#43,name#44,age#45,gender#46,birthday#47,zip#48,city#49] parquet

== Analyzed Logical Plan ==
cust_id: string, name: string, age: string, gender: string, birthday: string, zip

## `StorageLevel` Types:

(As of Spark `3.4`)

- `DISK_ONLY`: CPU efficient, memory efficient, slow to access, data is serialized when stored on disk
- `DISK_ONLY_2`: disk only, replicated 2x
- `DISK_ONLY_3`: disk only, replicated 3x

- `MEMORY_AND_DISK`: spills to disk if there's no space in memory
- `MEMORY_AND_DISK_2`: memory and disk, replicated 2x
- `MEMORY_AND_DISK_DESER`(default): same as `MEMORY_AND_DISK`, deserialized in both for fast access

- `MEMORY_ONLY`: CPU efficient, memory intensive
- `MEMORY_ONLY_2`: memory only, replicated 2x - for resilience, if one executor fails

**Note**: 
- `SER` is CPU intensive, memory saving as data is compact while `DESER` is CPU efficient, memory intensive
- Size of data on disk is lesser as data is in serialized format, while deserialized in memory as JVM objects for faster access

### When to use what?
```
Storage Level    Space used  CPU time  In memory  On-disk  Serialized
---------------------------------------------------------------------
MEMORY_ONLY          High        Low       Y          N        N         
MEMORY_ONLY_SER      Low         High      Y          N        Y     
MEMORY_AND_DISK      High        Medium    Some       Some     Some  
MEMORY_AND_DISK_SER  Low         High      Some       Some     Y     
DISK_ONLY            Low         High      N          Y        Y     
```

In [ ]:
df_base.unpersist()
df_base.persist(StorageLevel.MEMORY_ONLY)

df2 = (
    df_base
    .withColumn("test_column_1", lit("test_column_1"))
    .withColumn("birth_year", split("birthday", "/").getItem(2))
)

df1.show(5, False)

In [ ]:
spark.stop()